# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [13]:
import os
from google.colab import userdata

CLAUDE_API_KEY = userdata.get('claude_ironhack')
os.environ["ANTHROPIC_API_KEY"] = CLAUDE_API_KEY  # so pickt langchain_anthropic den Key automatisch auf

import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/zeroKool1ne/lab-chains-in-langchain/main/data/Data.csv')
df.head()
HUGGINGFACEHUB_API_TOKEN = os.getenv('hugging_face')

In [ ]:
#!pip install pandas

In [14]:
import pandas as pd

# github.com/DEINUSER/DEINFORK/blob/main/data/dataset.csv
# → raw.githubusercontent.com/DEINUSER/DEINFORK/main/data/dataset.csv

url = "https://raw.githubusercontent.com/zeroKool1ne/lab-chains-in-langchain/refs/heads/main/data/Data.csv"
df = pd.read_csv(url)
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


In [ ]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [1]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [15]:
!pip install -q langchain_community langchain-anthropic

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

# temperature=0.7: genug Kreativität für einen Produktnamen, aber nicht wild halluzinierend
llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0.7)

prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain = LLMChain(llm=llm, prompt=prompt)

product = df.Product[0]  # "Queen Size Sheet Set"
chain.run(product)

'# Best Names for a Queen Size Sheet Set Company\n\nHere are some strong options:\n\n## Direct/Descriptive\n- **Royal Linens** - suggests quality and the "royal" size\n- **Queen Bedding Co.** - straightforward and clear\n- **Premium Sheet Company** - emphasizes quality\n\n## More Creative\n- **Luxe Linens** - upscale, memorable\n- **Thread & Crown** - elegant, subtle reference to "queen"\n- **Serene Sheets** - focuses on comfort/sleep quality\n- **Woven Royalty** - combines product with size reference\n\n## Modern/Trendy\n- **Nest & Linen** - cozy, contemporary feel\n- **Threshold Home** - suggests quality standards\n- **Fiber & Form** - emphasizes craftsmanship\n\n## Key Considerations\n- **Avoid being too narrow** if you might expand product lines later\n- **Make it memorable** - something customers can easily recall and spell\n- **Consider your brand positioning** - luxury, eco-friendly, affordable, etc.\n\n**My top pick:** **Luxe Linens** or **Royal Linens** — they\'re professional

## SimpleSequentialChain

In [16]:
#Replace None by your own value and justify
from langchain_classic.chains import SequentialChain

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0.9)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")

second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")

third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followup_message")

overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

review = df.Review[5]
overall_chain(review)




> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': "# English Translation\n\nI find the taste mediocre. The mousse doesn't hold up, it's strange. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit!?",
 'summary': 'The reviewer found the product disappointing due to mediocre taste and poor mousse texture, suspecting it may be an old or counterfeit batch compared to store-bought versions.',
 'followup_message': "# Réponse de suivi\n\nMerci d'avoir partagé vos commentaires détaillés. Nous sommes vraiment désolés d'apprendre que votre expérience avec notre produit n'a pas été à la hauteur de vos attentes.\n\nVos observations concernant le goût médiocre et la texture de la mousse sont des préoccupations que nous prenons très au sérieux. Nous vous assurons que nous n'utilisons que des ingrédients de qualité supé

**Repeat the above twice for different products**

## SequentialChain

In [17]:
from langchain_classic.chains import SequentialChain

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0.9)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")

second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")

third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followup_message")

overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': "# Translation to English:\n\nI find the taste mediocre. The foam doesn't hold up, it's strange. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit!?",
 'summary': "The reviewer is dissatisfied with the product's mediocre taste and poor foam quality, suspecting it may be an old or counterfeit batch compared to store-bought versions.",
 'followup_message': "# Réponse de suivi\n\nMerci beaucoup pour votre retour détaillé. Nous sommes vraiment désolés d'apprendre que vous avez eu une expérience décevante avec notre produit concernant le goût et la qualité de la mousse.\n\nVos préoccupations sont tout à fait légitimes, et nous prenons très au sérieux la possibilité que vous ayez reçu un lot ancien ou contrefait. Cela ne correspond absolument pas aux standards 

**Repeat the above twice for different products or reviews**

## Router Chain

In [18]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(router_chain=router_chain,
                          destination_chains=destination_chains,
                          default_chain=default_chain, verbose=True)

chain.run("What is black body radiation?")
chain.run("what is 2 + 2")
chain.run("Why does every cell in our body contain DNA?")
chain.run("What triggered the fall of the Berlin Wall?")  # eigenes kreatives Beispiel für die "be creative"-Anforderung

NameError: name 'prompt_infos' is not defined

**Repeat the above at least once for different inputs and chains executions - Be creative!**